# 5. CnnVisual

Visualisasi hasil CNN dari artefak yang sudah tersimpan.


In [ ]:
from pathlib import Path
import sys

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    raise RuntimeError('root repo tidak ditemukan')

ROOT = find_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('root:', ROOT)


In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TABLE_DIR = ROOT / 'reports/tables'
FIG_DIR = ROOT / 'reports/figures'
CNN_MODEL_DIR = ROOT / 'models/cnn'
FIG_DIR.mkdir(parents=True, exist_ok=True)

cnn = pd.read_csv(TABLE_DIR / 'cnn_records.csv')
manual = pd.read_csv(TABLE_DIR / 'cnn_manual_comparison.csv')
print('cnn:', len(cnn), 'manual:', len(manual))


In [ ]:
plot_df = cnn.sort_values('experiment')
plt.figure(figsize=(10, 4))
plt.bar(plot_df['experiment'].astype(str), plot_df['macro_f1'])
plt.xlabel('Experiment')
plt.ylabel('Macro F1')
plt.title('CNN Macro F1')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_macro_f1.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, col, title in [
    (axes[0, 0], 'num_layers', 'Layer'),
    (axes[0, 1], 'filters', 'Filter'),
    (axes[1, 0], 'kernel_size', 'Kernel'),
    (axes[1, 1], 'pooling', 'Pooling'),
]:
    part = cnn.groupby(col)['macro_f1'].mean().reset_index()
    ax.bar(part[col].astype(str), part['macro_f1'])
    ax.set_title(title)
    ax.set_ylabel('Macro F1')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_hyperparameter_effects.png', dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
for exp_id in sorted(cnn['experiment'].astype(int).unique()):
    path = CNN_MODEL_DIR / f'history_exp_{exp_id}.pkl'
    if not path.exists():
        continue
    with open(path, 'rb') as file:
        history = pickle.load(file)
    values = history.get('loss', [])
    if values:
        plt.plot(values, alpha=0.45, label=f'exp {exp_id}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Training Loss Semua Eksperimen')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_training_loss.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
for exp_id in sorted(cnn['experiment'].astype(int).unique()):
    path = CNN_MODEL_DIR / f'history_exp_{exp_id}.pkl'
    if not path.exists():
        continue
    with open(path, 'rb') as file:
        history = pickle.load(file)
    values = history.get('val_loss', [])
    if values:
        plt.plot(values, alpha=0.45, label=f'exp {exp_id}')
plt.xlabel('Epoch')
plt.ylabel('Val loss')
plt.title('CNN Validation Loss Semua Eksperimen')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_validation_loss.png', dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
labels = manual['implementation'].astype(str)
values = manual['macro_f1'].fillna(manual.get('macro_f1_small_batch', np.nan))
plt.bar(labels, values)
plt.xticks(rotation=20, ha='right')
plt.ylabel('Macro F1')
plt.title('CNN Keras vs Scratch')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_keras_vs_scratch.png', dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(labels, manual['params'])
plt.xticks(rotation=20, ha='right')
plt.ylabel('Parameter')
plt.title('Shared vs Non-shared Parameter')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cnn_parameter_count.png', dpi=150)
plt.show()
